# Qwen-Image-2512 Character LoRA — Google Colab Training

Train a character-specific LoRA on Qwen-Image-2512 (20B MMDiT) directly in Google Colab.

**Recommended runtime:** A100 (80 GB) or H100 — minimum 40 GB VRAM required. T4 is insufficient.

Run cells top-to-bottom. Every section is idempotent — if your runtime disconnects, reconnect, re-run all cells, and training resumes from the latest checkpoint automatically.

## Before You Start

### 1. Colab Secret: `HF_TOKEN`
Go to **Secrets** (key icon in the left sidebar) and add:
- Name: `HF_TOKEN`
- Value: your HuggingFace access token with read access to `Qwen/Qwen-Image-2512`
- Enable **Notebook access**

### 2. Google Drive: training dataset
Put your captioned subject images zip at:
```
MyDrive/lora-trainer/input/batch1.zip
```
The zip should contain images + matching `.txt` caption files (one `.txt` per image, same filename).
Use `scripts/caption.py` from the repo to generate captions if you haven't yet.

### Directory layout (for reference)
```
/content/
├── drive/MyDrive/lora-trainer/
│   ├── input/
│   │   └── batch1.zip          ← YOUR DATASET (put here before running)
│   └── output/
│       └── qwen-batch1/        ← LoRA weights saved here (persists after disconnect)
│
└── data/                       ← ephemeral working dir (lost on runtime reset)
    ├── models/Qwen-Image-2512/ ← ~40 GB base model
    ├── dataset/
    │   ├── images/             ← your subject images + .txt captions
    │   └── regularization/
    │       └── images/         ← regularization dataset (auto-downloaded)
    ├── cache/                  ← SimpleTuner VAE + text embed cache
    └── config/                 ← generated config files
```

In [ ]:
# ── User Config ────────────────────────────────────────────────────────────────
# Edit these values before running the notebook.

TRIGGER_WORD        = "ohwx"          # Trigger word embedded in all captions
LORA_OUTPUT_NAME    = "qwen-batch1"   # Subfolder name under Drive output dir
MAX_TRAIN_STEPS     = 3000
TRAIN_BATCH_SIZE    = 1               # A100 80 GB can handle 2; H100 can handle 2-4
LORA_RANK           = 64
LORA_ALPHA          = 64
LEARNING_RATE       = 3e-5

# Attention mechanism — choose based on your GPU:
#   "flash_attention_2"  → A100, H100 (recommended for Colab)
#   "sdpa"               → any GPU, safe fallback with lower throughput
#   "flash-attn-3"       → Blackwell only (GB10/GB200), NOT available on Colab
ATTENTION_MECHANISM = "flash_attention_2"

# Paths on Google Drive
DATASET_ZIP_DRIVE_PATH = f"/content/drive/MyDrive/lora-trainer/input/batch1.zip"
OUTPUT_DRIVE_DIR       = f"/content/drive/MyDrive/lora-trainer/output/{LORA_OUTPUT_NAME}"

# Internal working paths (ephemeral, under /content/data)
DATA_ROOT            = "/content/data"
MODEL_DIR            = f"{DATA_ROOT}/models/Qwen-Image-2512"
SUBJECT_IMAGES_DIR   = f"{DATA_ROOT}/dataset/images"
REG_IMAGES_DIR       = f"{DATA_ROOT}/dataset/regularization/images"
CONFIG_DIR           = f"{DATA_ROOT}/config"
VAE_CACHE_SUBJECT    = f"{DATA_ROOT}/cache/vae/subject"
VAE_CACHE_REG        = f"{DATA_ROOT}/cache/vae/regularization"
TEXT_CACHE_DIR       = f"{DATA_ROOT}/cache/text/qwen_image"
# ──────────────────────────────────────────────────────────────────────────────

print("Config loaded.")
print(f"  Output name : {LORA_OUTPUT_NAME}")
print(f"  Train steps : {MAX_TRAIN_STEPS}")
print(f"  Batch size  : {TRAIN_BATCH_SIZE}")
print(f"  Attention   : {ATTENTION_MECHANISM}")

## Section 0: Runtime Check
Verify a GPU is attached and print its specs.

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU found. Change runtime type to GPU (A100 or H100).")

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

import torch
assert torch.cuda.is_available(), "CUDA not available — check runtime type"
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

gpu_name = gpu_info.split(',')[0].strip().lower()
if not any(g in gpu_name for g in ['a100', 'h100', 'h200']):
    print(f"\nWARNING: GPU '{gpu_info.split(chr(44))[0].strip()}' may not have enough VRAM.")
    print("Recommended: A100 (80 GB) or H100. Minimum 40 GB VRAM required.")
else:
    print("GPU OK.")

## Section 1: Mount Google Drive
Mounts Drive and verifies your dataset zip is present.

In [ ]:
import os
from pathlib import Path

# Mount Drive (idempotent)
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

# Verify dataset zip exists
zip_path = Path(DATASET_ZIP_DRIVE_PATH)
if not zip_path.exists():
    raise FileNotFoundError(
        f"Dataset zip not found at {zip_path}\n"
        "Please upload batch1.zip to MyDrive/lora-trainer/input/ and re-run."
    )
print(f"Dataset zip found: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

# Create output directory on Drive
Path(OUTPUT_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DRIVE_DIR}")

### Optional: Cache model weights on Drive

The base model (~40 GB) is re-downloaded every time you get a fresh runtime. If you want to avoid this, run the cell below **once** to symlink the model directory to Google Drive. Subsequent runtimes will find the weights already there.

**Trade-off:** Drive I/O is slower than Colab's local SSD. Training startup will take a few extra minutes.

**Skip this cell if you prefer a fresh download each time (faster during training).**

In [ ]:
# OPTIONAL — run this cell to cache the model on Drive
# Skip entirely if you want the model on fast local SSD instead.

import os
from pathlib import Path

drive_model_cache = "/content/drive/MyDrive/lora-trainer/model-cache/Qwen-Image-2512"
Path(drive_model_cache).mkdir(parents=True, exist_ok=True)

local_model_parent = Path(MODEL_DIR).parent
local_model_parent.mkdir(parents=True, exist_ok=True)

if Path(MODEL_DIR).is_symlink():
    print(f"Symlink already exists: {MODEL_DIR} -> {os.readlink(MODEL_DIR)}")
elif Path(MODEL_DIR).exists():
    print(f"{MODEL_DIR} already exists as a real directory (not symlinking).")
else:
    os.symlink(drive_model_cache, MODEL_DIR)
    print(f"Symlinked {MODEL_DIR} -> {drive_model_cache}")
    print("Model weights will be cached to Drive and reused across runtimes.")

## Section 2: Install Dependencies
Installs SimpleTuner from git main (required for Qwen-Image batch>1 fix). Skips if already installed.

In [ ]:
import importlib.util, subprocess, sys

def run(cmd, **kwargs):
    """Run a shell command, streaming output, raising on failure."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# Install SimpleTuner if not already present
if importlib.util.find_spec("simpletuner") is None:
    print("Installing SimpleTuner from git main...")
    run("pip install -q 'simpletuner[cuda] @ git+https://github.com/bghira/SimpleTuner.git@main'")
    print("SimpleTuner installed.")
else:
    print("SimpleTuner already installed, skipping.")

# Apply safety_check.py patch (handles nvidia-smi returning [N/A] on unified-memory GPUs).
# On Colab discrete GPUs this is a no-op functionally, but keeps parity with the Dockerfile.
import simpletuner, os
safety_check_path = os.path.join(
    os.path.dirname(simpletuner.__file__),
    "helpers", "training", "default_settings", "safety_check.py"
)
old_line = "total_memory = int(output.decode().strip()) / 1024"
new_line  = "raw = output.decode().strip(); total_memory = (int(raw) if raw.lstrip('-').isdigit() else 131072) / 1024"

with open(safety_check_path, 'r') as f:
    content = f.read()

if old_line in content:
    with open(safety_check_path, 'w') as f:
        f.write(content.replace(old_line, new_line))
    print("safety_check.py patched.")
else:
    print("safety_check.py already patched or line not found (skipping).")

print("Dependencies ready.")

## Section 3: Prepare Dataset
Extracts subject images from Drive zip and downloads regularization images from HuggingFace. Both steps are idempotent.

In [ ]:
import os, zipfile
from pathlib import Path
from huggingface_hub import snapshot_download

# ── Subject images ──────────────────────────────────────────────────────────
subject_dir = Path(SUBJECT_IMAGES_DIR)
subject_dir.mkdir(parents=True, exist_ok=True)

existing_images = list(subject_dir.rglob('*.jpg')) + list(subject_dir.rglob('*.png')) + list(subject_dir.rglob('*.jpeg'))
if existing_images:
    print(f"Subject images already extracted ({len(existing_images)} images). Skipping unzip.")
else:
    print(f"Extracting {DATASET_ZIP_DRIVE_PATH} ...")
    with zipfile.ZipFile(DATASET_ZIP_DRIVE_PATH, 'r') as zf:
        zf.extractall(str(subject_dir))
    images_after = list(subject_dir.rglob('*.jpg')) + list(subject_dir.rglob('*.png')) + list(subject_dir.rglob('*.jpeg'))
    print(f"Extracted {len(images_after)} images to {subject_dir}")

# Verify captions exist
all_images = list(subject_dir.rglob('*.jpg')) + list(subject_dir.rglob('*.png')) + list(subject_dir.rglob('*.jpeg'))
missing_captions = [img for img in all_images if not img.with_suffix('.txt').exists()]
if missing_captions:
    print(f"WARNING: {len(missing_captions)} images have no .txt caption file:")
    for p in missing_captions[:5]:
        print(f"  {p}")
    if len(missing_captions) > 5:
        print(f"  ... and {len(missing_captions) - 5} more")
else:
    print(f"All {len(all_images)} images have captions. Subject dataset ready.")

# ── Regularization images ───────────────────────────────────────────────────
reg_dir = Path(REG_IMAGES_DIR)
reg_dir.mkdir(parents=True, exist_ok=True)

reg_images = list(reg_dir.rglob('*.jpg')) + list(reg_dir.rglob('*.png'))
if reg_images:
    print(f"Regularization images already present ({len(reg_images)} images). Skipping download.")
else:
    print("Downloading bghira/pseudo-camera-10k regularization dataset...")
    print("This may take several minutes.")
    snapshot_download(
        repo_id="bghira/pseudo-camera-10k",
        repo_type="dataset",
        local_dir=str(reg_dir),
    )
    reg_images_after = list(reg_dir.rglob('*.jpg')) + list(reg_dir.rglob('*.png'))
    print(f"Downloaded {len(reg_images_after)} regularization images.")

print("Dataset preparation complete.")

## Section 4: Download Base Model
Downloads Qwen/Qwen-Image-2512 (~40 GB) from HuggingFace. Uses your `HF_TOKEN` Colab Secret. Idempotent — skips if already downloaded.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download
from google.colab import userdata

# Read HF token from Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found in Colab Secrets.\n"
        "Go to the key icon in the left sidebar, add HF_TOKEN, and enable Notebook access."
    )
print("HF_TOKEN loaded from Colab Secrets.")

model_dir = Path(MODEL_DIR)

# Idempotency check: presence of config.json in the model directory
if (model_dir / 'config.json').exists():
    print(f"Model already present at {model_dir}. Skipping download.")
else:
    model_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading Qwen/Qwen-Image-2512 to {model_dir} ...")
    print("This takes ~10-15 minutes on a fresh Colab runtime (~40 GB).")
    snapshot_download(
        repo_id="Qwen/Qwen-Image-2512",
        local_dir=str(model_dir),
        token=HF_TOKEN,
    )
    print("Model download complete.")

# Store token in env for SimpleTuner's hub calls
os.environ['HF_TOKEN'] = HF_TOKEN
print("Base model ready.")

## Section 5: Write Config Files
Generates all 4 SimpleTuner config files from your config variables. Always regenerates to stay in sync.

In [ ]:
import json, os
from pathlib import Path

config_dir = Path(CONFIG_DIR)
config_dir.mkdir(parents=True, exist_ok=True)

# ── config.json ─────────────────────────────────────────────────────────────
config_json = {
    "model_family": "qwen_image",
    "model_flavour": "v1.0",
    "model_type": "lora",
    "pretrained_model_name_or_path": MODEL_DIR,
    "base_model_precision": "no_change",
    "mixed_precision": "bf16",
    "lora_type": "standard",
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler": "constant_with_warmup",
    "lr_warmup_steps": 100,
    "optimizer": "optimi-lion",
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_checkpointing": True,
    "max_train_steps": MAX_TRAIN_STEPS,
    "num_train_epochs": 0,
    "max_grad_norm": 0.01,
    "seed": 42,
    "flow_schedule_shift": 1.73,
    "resolution": 1024,
    "resolution_type": "pixel_area",
    "minimum_image_size": 0,
    "caption_dropout_probability": 0.0,
    "data_backend_config": f"{CONFIG_DIR}/multidatabackend.json",
    "output_dir": OUTPUT_DRIVE_DIR,
    "checkpoint_step_interval": 500,
    "checkpoints_total_limit": 5,
    "resume_from_checkpoint": "latest",
    "ignore_final_epochs": True,
    "disable_bucket_pruning": True,
    "validation_steps": 1000,
    "validation_guidance": 4.0,
    "validation_guidance_rescale": 0.0,
    "validation_lycoris_strength": 1.0,
    "validation_num_inference_steps": 20,
    "validation_prompt": f"A photo of {TRIGGER_WORD} standing against a white background, natural lighting",
    "validation_prompt_library": False,
    "user_prompt_library": f"{CONFIG_DIR}/user_prompt_library.json",
    "validation_negative_prompt": "ugly, cropped, blurry, low-quality, deformed, disfigured",
    "validation_resolution": "1024x1024",
    "validation_seed": 42,
    "num_eval_images": 1,
    "report_to": "none",
    "tracker_project_name": "qwen-character-lora",
    "tracker_run_name": f"{TRIGGER_WORD}-training",
    "push_to_hub": False,
    "push_checkpoints_to_hub": False,
    "use_ema": False,
    "vae_batch_size": 1,
    "compress_disk_cache": False,
    "attention_mechanism": ATTENTION_MECHANISM,
    "disable_benchmark": False,
    "skip_file_discovery": False
}

(config_dir / 'config.json').write_text(json.dumps(config_json, indent=4))
print("Wrote config.json")

# ── multidatabackend.json ────────────────────────────────────────────────────
multidatabackend = [
    {
        "id": "subject-images",
        "type": "local",
        "instance_data_dir": SUBJECT_IMAGES_DIR,
        "caption_strategy": "textfile",
        "crop": False,
        "resolution": 1024,
        "resolution_type": "pixel_area",
        "minimum_image_size": 512,
        "maximum_image_size": 1328,
        "target_downsample_size": 1024,
        "cache_dir_vae": VAE_CACHE_SUBJECT,
        "metadata_backend": "discovery",
        "repeats": 20,
        "is_regularisation_data": False,
        "disabled": False,
        "skip_file_discovery": ""
    },
    {
        "id": "regularization-images",
        "type": "local",
        "instance_data_dir": REG_IMAGES_DIR,
        "caption_strategy": "instanceprompt",
        "instance_prompt": "a photo of a person",
        "crop": True,
        "crop_style": "center",
        "crop_aspect": "square",
        "resolution": 1024,
        "resolution_type": "pixel_area",
        "minimum_image_size": 512,
        "maximum_image_size": 1328,
        "target_downsample_size": 1024,
        "cache_dir_vae": VAE_CACHE_REG,
        "metadata_backend": "discovery",
        "repeats": 0,
        "is_regularisation_data": True,
        "disabled": False,
        "skip_file_discovery": ""
    },
    {
        "id": "text-embeds",
        "type": "local",
        "dataset_type": "text_embeds",
        "default": True,
        "cache_dir": TEXT_CACHE_DIR,
        "write_batch_size": 16,
        "disabled": False
    }
]

(config_dir / 'multidatabackend.json').write_text(json.dumps(multidatabackend, indent=4))
print("Wrote multidatabackend.json")

# ── user_prompt_library.json ─────────────────────────────────────────────────
# Prompts reference TRIGGER_WORD; substituted from configmap values.
user_prompts = {
    "identity_check": f"A photo of {TRIGGER_WORD} standing against a plain white background, arms at sides, looking directly at the camera, neutral expression, soft even studio lighting",
    "pose_sitting": f"A photo of {TRIGGER_WORD} sitting cross-legged on a wooden floor, hands resting on knees, relaxed expression, warm indoor lighting",
    "pose_action": f"A photo of {TRIGGER_WORD} running through a grassy park, mid-stride, energetic expression, bright daylight with dappled shadows",
    "setting_urban": f"A photo of {TRIGGER_WORD} standing on a busy city sidewalk at night, neon signs reflected on wet pavement, wearing a dark overcoat",
    "setting_nature": f"A photo of {TRIGGER_WORD} walking through a snowy pine forest at golden hour, breath visible in cold air, wearing a red winter jacket",
    "outfit_formal": f"A photo of {TRIGGER_WORD} wearing a tailored navy blue suit and white dress shirt, standing in a modern office with floor-to-ceiling windows",
    "outfit_casual": f"A photo of {TRIGGER_WORD} wearing a plain white t-shirt and jeans, sitting on a park bench, relaxed posture, sunny day",
    "close_up": f"A close-up photo of {TRIGGER_WORD} from the shoulders up, looking slightly to the left, gentle smile, soft natural window light"
}

(config_dir / 'user_prompt_library.json').write_text(json.dumps(user_prompts, indent=4))
print("Wrote user_prompt_library.json")

# ── config.env ───────────────────────────────────────────────────────────────
config_env = """export TRAINING_NUM_PROCESSES=1
export TRAINING_NUM_MACHINES=1
export TRAINING_DYNAMO_BACKEND=no
export MIXED_PRECISION=bf16
export SIMPLETUNER_LOG_LEVEL=INFO
"""
(config_dir / 'config.env').write_text(config_env)
print("Wrote config.env")

print(f"\nAll config files written to {config_dir}")

## Section 6: Train

Runs SimpleTuner training. Output streams live to this cell.

**If your runtime disconnects:** reconnect, re-run all cells from the top, then re-run this cell. Training will resume automatically from the latest checkpoint (via `resume_from_checkpoint: latest`).

Expected duration: ~2-4 hours for 3000 steps with 40-80 subject images on an A100.

In [ ]:
import os, multiprocessing, logging, logging.config

# Set environment variables (mirrors config.env and train-job.yaml)
os.environ.update({
    'TRAINING_NUM_PROCESSES': '1',
    'TRAINING_NUM_MACHINES': '1',
    'TRAINING_DYNAMO_BACKEND': 'no',
    'MIXED_PRECISION': 'bf16',
    'SIMPLETUNER_LOG_LEVEL': 'INFO',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    # Tell SimpleTuner to use dict-based config rather than argparse
    'CONFIG_BACKEND': 'cmd',
    'ENV': 'default',
})

# Quiet noisy third-party loggers before importing SimpleTuner
logging.getLogger('DeepSpeed').setLevel('ERROR')
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel('ERROR')
logging.config.dictConfig({'version': 1, 'disable_existing_loggers': False})
os.environ['ACCELERATE_LOG_LEVEL'] = 'WARNING'

from simpletuner.helpers.configuration.json_file import normalize_args
from simpletuner.helpers.training.state_tracker import StateTracker
from simpletuner.helpers.training.trainer import Trainer

# Point SimpleTuner at the config directory written in Section 5
StateTracker.set_config_path(f'{CONFIG_DIR}/')

# Load the same config dict used to write config.json — no file re-read needed
loaded_config = normalize_args(config_json)

try:
    multiprocessing.set_start_method('fork')
except Exception as e:
    print(f'Warning: could not set fork start method: {e}')

print(f'Output: {OUTPUT_DRIVE_DIR}')
print('Starting training...\n')

trainer = Trainer(loaded_config, exit_on_error=True)
trainer.configure_webhook()
trainer.init_noise_schedule()
trainer.init_seed()
trainer.init_huggingface_hub()
trainer.init_preprocessing_models()
trainer.init_precision(preprocessing_models_only=True)
trainer.init_data_backend()
trainer.init_unload_text_encoder()
trainer.init_unload_vae()
trainer.init_load_base_model()
trainer.init_delete_model_caches()
trainer.init_controlnet_model()
trainer.init_tread_model()
trainer.init_precision()
trainer.init_freeze_models()
trainer.init_trainable_peft_adapter()
trainer.init_ema_model()
trainer.init_precision(ema_only=True)
trainer.move_models(destination='accelerator')
trainer.init_distillation()
trainer.init_validations()
trainer.init_benchmark_base_model()
trainer.resume_and_prepare()
trainer.init_trackers()
trainer.train()

print('\nTraining complete!')

## Section 7: Save Outputs

Saves the trained LoRA weights to Google Drive and offers a browser download.

The weights are already being written directly to Drive during training (via `output_dir`). This cell copies them to a dated subfolder for archiving and triggers a browser download as backup.

In [ ]:
import zipfile, os
from pathlib import Path
from google.colab import files

output_dir = Path(OUTPUT_DRIVE_DIR)

# Check training produced output
safetensors = list(output_dir.rglob('*.safetensors'))
if not safetensors:
    print(f"No .safetensors files found in {output_dir}.")
    print("Training may not have completed. Check the training cell output.")
else:
    print(f"Found {len(safetensors)} .safetensors file(s):")
    for f_path in safetensors:
        size_mb = f_path.stat().st_size / 1e6
        print(f"  {f_path.relative_to(output_dir)} ({size_mb:.1f} MB)")

    print(f"\nOutputs already saved to Drive: {output_dir}")

    # Zip for browser download
    zip_path = Path(f"/content/{LORA_OUTPUT_NAME}.zip")
    print(f"\nCreating zip for download: {zip_path}")
    with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
        for f_path in safetensors:
            zf.write(str(f_path), f_path.relative_to(output_dir))
    print(f"Zip created ({zip_path.stat().st_size / 1e6:.1f} MB). Starting download...")
    files.download(str(zip_path))

    print("\nDone! Your LoRA weights are at:")
    print(f"  Drive: {output_dir}")
    print(f"  Local zip: {zip_path}")